In [ ]:
# Optional: mount Google Drive on Colab
try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    pass  # skip outside Colab


In [ ]:
from typing import Literal
import numpy as np
import torch
import pandas as pd

class SudokuTable():
    UNKNOWN_CHAR = '*'

    def __init__(self, string: str = None) -> None:
        self.initial = string
        self.grid = [[0] * 9 for _ in range(9)]
        if string is not None:
            self.grid = self.to_grid(string)
        self.dofs = self.update_dofs()

    def candidates(self, row: int, col: int) -> set[int]:
        '''
            Returns a set of valid digits to fill in the given position
        '''
        used = set()
        for i in range(9):                          # go through the column
            if i != row:
                used.add(self.grid[i][col])      
        for j in range(9):                          # go through the row
            if j != col:
                used.add(self.grid[row][j])

        block_row = row // 3
        block_col = col // 3
        for i in range(block_row * 3, block_row * 3 + 3):
            for j in range(block_col * 3, block_col * 3 + 3):
                if i != row and j != col:
                    used.add(self.grid[i][j])
        
        used.discard(0)
        candidates = set(range(1, 10)) - used

        return candidates
    
    def dof(self, row: int, col: int) -> int:
        '''
            Returns the DOF of the position
        '''
        return len(self.candidates(row, col))

    def update_dofs(self):
        dofs = [[0] * 9 for _ in range(9)]
        for i in range(9):
            for j in range(9):
                if self.grid[i][j] == 0:
                    dofs[i][j] = self.dof(i, j)
                                    # -1 means its position is filled with some number. Calculating its dof doesn't have meanings.
                else:
                    dofs[i][j] = -1 

        return dofs

    def to_string(self, row_sep=None) -> str:
        result = ""
        for i in range(9):
            for j in range(9):
                result += str(self.grid[i][j])
            if i < 8 and row_sep is not None:
                result += row_sep
        return result
    
    def to_grid(self, string: str) -> list[list[int]]:
        for i in range(9):
            for j in range(9):
                self.grid[i][j] = int(string[i * 9 + j])
        return self.grid

    def num_locations_with(self, num_candidates) -> int:
        return sum(value == num_candidates for row in self.dofs for value in row)
    
    def get_coords_of(self, num_candidates) -> list[tuple]:

        positions = []
        for i in range(9):
            for j in range(9):
                if self.dofs[i][j] == num_candidates:
                    positions.append((i, j))

        return positions
    
    def to_tensor(self, to_one_hot=True) -> torch.Tensor:
        tensor = torch.tensor(self.grid, dtype=torch.long)
        if to_one_hot:
            # (10, 9, 9) one-hot, same layout as SudokuDataset
            return torch.nn.functional.one_hot(tensor, num_classes=10).permute(2, 0, 1).float()
        return tensor.float()

    def is_valid(self) -> bool:
        for i in range(9):
            seen = set()
            for item in self.grid[i]:
                if item != 0:
                    if item in seen:
                        return False
                    seen.add(item)

        for j in range(9):
            seen = set()
            for i in range(9):
                item = self.grid[i][j]
                if item != 0:
                    if item in seen:
                        return False
                    seen.add(item)

        for bi in range(0, 9, 3):
            for bj in range(0, 9, 3):
                seen = set()
                for i in range(bi, bi+3):
                    for j in range(bj, bj+3):
                        item = self.grid[i][j]
                        if item != 0:
                            if item in seen:
                                return False
                            seen.add(item)

        return True
    
    def is_complete(self) -> bool:
        for i in range(9):
            for j in range(9):
                if self.grid[i][j] == 0:
                    return False
        return True
    
    def step(self, row: int, col: int, value: int):

        if self.grid[row][col] != 0:      # Already filled
            return self.to_string(), -0.1, False, {'valid': False}
        else:
            self.grid[row][col] = value
            if self.is_valid():
                self.dofs = self.update_dofs()
                if self.is_complete():
                    return self.to_string(), 1.0, True, {'valid': True, 'solved': True}
                else:
                    return self.to_string(), 0.01, False, {'valid': True}
            else:
                self.grid[row][col] = 0   # If is a illegal move, undo it
                return self.to_string(), -0.1, False, {'valid': False}
            

    def show(self, format: Literal["plain", "boxed", "medium"] = "medium") -> str:
        result = ""
        if format == "plain":
            for i in range(9):
                for j in range(9):
                    result += str(self.grid[i][j]) if self.grid[i][j] != 0 else self.UNKNOWN_CHAR
                    if j < 8:
                        result += ' '
                result += '\n'
        
        elif format == "boxed":
            result += "┌───────────────────┐\n"
            for i in range(9):
                result += '│ '
                for j in range(9):
                    result += str(self.grid[i][j]) if self.grid[i][j] != 0 else self.UNKNOWN_CHAR
                    result += ' '
                result += '│\n'
            result += "└───────────────────┘\n"

        elif format == "medium":
            result +=         "╔═══════╤═══════╤═══════╗\n"
            for i in range(9):
                if i == 3 or i == 6:
                    result += "╟───────┼───────┼───────╢\n"

                for j in range(9):
                    if j == 0:
                        result += '║ '
                    elif j == 3 or j == 6:
                        result += '│ '
                    result += str(self.grid[i][j]) if self.grid[i][j] != 0 else self.UNKNOWN_CHAR
                    result += ' '
                result += '║\n'

            result += "╚═══════╧═══════╧═══════╝\n"
        
        return result
    
    def __repr__(self) -> str:
        return self.show(format="medium")


In [2]:
import pandas as pd

# Preview only; pass dataset to *Train functions.
df = pd.read_csv("data/sudoku.csv")


df


,quizzes,solutions
0,0043002090050090010700600430060020871900074000...,8643712593258497619712658434361925871986574322...
1,0401000501070039605200080000000000170009068008...,3461792581875239645296483719658324174729168358...
2,6001203840084590720000060050002640300700800069...,6951273841384596727248369158512647392739815469...
3,4972000001004000050000160986203000403009000000...,4972583161864397252537164986293815473759641828...
4,0059103080094030600275001000300002010008200070...,4659123781894735623275681497386452919548216372...
...,...,...
999995,3000280000290000300054001077402030980086070031...,3175289464291768356854391277462135989586472131...
999996,0030006000040860057000009409350407208067200502...,5234976811942863757685139429356417288167294532...
999997,0003508200618040300500090000700600029030070100...,7493568212618745393582197468749613529235876146...
999998,0702006900030400010000650205600300000947005800...,4752816936239478511893657245628341793947165828...


In [ ]:
from torch.utils.data import Dataset
import torch.nn as nn


def normalize_puzzle_string(s: str) -> str:
    """Map blank cells to '0' (Kaggle 1m) or normalize '.' (3m)."""
    return str(s).replace(".", "0")


class SudokuDataset(Dataset):
    def __init__(self, puzzles: list[str], solutions: list[str]):
        self.input = puzzles
        self.target = solutions

    def __len__(self):
        return len(self.input)

    def __getitem__(self, index):   # -> (N, C, H, W)

        p_str = normalize_puzzle_string(self.input[index])
        s_str = self.target[index]
        
        p_ints = torch.tensor([int(c) for c in p_str], dtype=torch.long).view(9, 9)
        s_ints = torch.tensor([int(c) for c in s_str], dtype=torch.long).view(9, 9) - 1

        p_one_hot = torch.nn.functional.one_hot(p_ints, num_classes=10).permute(2, 0, 1).float()
        s_one_hot = torch.nn.functional.one_hot(s_ints, num_classes=9).permute(2, 0, 1).float()

        return (p_one_hot, s_one_hot)

        


In [6]:
class SudokuCNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(

            nn.Conv2d(in_channels=10, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(

            nn.Conv2d(in_channels=256, out_channels=128, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=128, out_channels=64, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=9, kernel_size=1)
        )

    def forward(self, x):
        features = self.encoder(x)
        logits = self.decoder(features)
        
        return logits


In [ ]:
class SudokuMLPModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_features=9*9*10, out_features=256),
            nn.Linear(in_features=256, out_features=256),
            nn.Linear(in_features=256, out_features=729)
        )
    def forward(self, x):
        batch_size = x.size(0)
        x = x.view(batch_size, -1)
        x = self.mlp(x)
        x = x.view(batch_size, 9, 9, 9).permute(0, 3, 1, 2)
        return x


In [ ]:
def get_constraints(board):
    """board (B,10,9,9) -> constraint tensor (B,37,9,9)."""
    if board.dim() == 3:
        board = board.unsqueeze(0)
    B, _, _, _ = board.shape
    board_digits = board.argmax(dim=1)
    constraints = torch.zeros(B, 37, 9, 9, dtype=torch.float32, device=board.device)
    for b in range(B):
        bd = board_digits[b]
        for i in range(9):
            for j in range(9):
                row_digits = bd[i, :]
                for d in range(1, 10):
                    if d in row_digits:
                        constraints[b, d - 1, i, j] = 1.0
                col_digits = bd[:, j]
                for d in range(1, 10):
                    if d in col_digits:
                        constraints[b, 9 + (d - 1), i, j] = 1.0
                box_row, box_col = (i // 3) * 3, (j // 3) * 3
                box_digits = bd[box_row:box_row + 3, box_col:box_col + 3].flatten()
                for d in range(1, 10):
                    if d in box_digits:
                        constraints[b, 18 + (d - 1), i, j] = 1.0
                digit = int(bd[i, j].item())
                constraints[b, 27 + digit, i, j] = 1.0
    return constraints


In [ ]:
class SudokuCNNConstraintModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.encoder = nn.Sequential(

            nn.Conv2d(in_channels=37, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(

            nn.Conv2d(in_channels=256, out_channels=128, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=128, out_channels=64, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=9, kernel_size=1)
        )

    def forward(self, x):
        x = get_constraints(x)
        features = self.encoder(x)
        logits = self.decoder(features)
        
        return logits


In [ ]:
class SudokuConstraintMLPModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_features=81*37, out_features=1024),
            nn.Linear(in_features=1024, out_features=1024),
            nn.Linear(in_features=1024, out_features=729)
        )
    def forward(self, x):
        x = get_constraints(x)
        B = x.size(0)
        x = x.view(B, -1)
        x = self.mlp(x)
        x = x.view(B, 9, 9, 9).permute(0, 3, 1, 2)
        return x


In [ ]:
def build_sudoku_norm_adj(dtype=torch.float32):
    """Build normalized Sudoku peer adjacency (81 nodes)."""
    N = 81
    idx = torch.arange(N)
    r = idx // 9
    c = idx % 9
    br = r // 3
    bc = c // 3
    ri, rj = r.unsqueeze(1), r.unsqueeze(0)
    ci, cj = c.unsqueeze(1), c.unsqueeze(0)
    bri, brj = br.unsqueeze(1), br.unsqueeze(0)
    bci, bcj = bc.unsqueeze(1), bc.unsqueeze(0)
    peer = (ri == rj) | (ci == cj) | ((bri == brj) & (bci == bcj))
    A = peer.to(dtype=dtype)
    deg = A.sum(dim=1).clamp(min=1.0)
    d_inv_sqrt = deg.pow(-0.5)
    Dm = torch.diag(d_inv_sqrt)
    return Dm @ A @ Dm


class SudokuGNNModel(nn.Module):
    """GCN message passing with residual updates; (B,10,9,9) -> (B,9,9,9) logits."""

    def __init__(self, hidden_dim: int = 128, num_layers: int = 6):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.register_buffer("adj", build_sudoku_norm_adj())
        self.input_lin = nn.Linear(10, hidden_dim)
        self.layers = nn.ModuleList(
            [nn.Linear(hidden_dim, hidden_dim) for _ in range(num_layers)]
        )
        self.norms = nn.ModuleList(
            [nn.LayerNorm(hidden_dim) for _ in range(num_layers)]
        )
        self.out_lin = nn.Linear(hidden_dim, 9)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 10, 9, 9)
        B = x.size(0)
        h = x.permute(0, 2, 3, 1).reshape(B, 81, 10)
        h = self.input_lin(h)
        adj = self.adj.to(dtype=h.dtype, device=h.device)
        for lin, ln in zip(self.layers, self.norms):
            msg = torch.einsum("ij,bjf->bif", adj, h)
            msg = ln(msg)
            h = h + torch.relu(lin(msg))
        logits = self.out_lin(h)
        logits = logits.view(B, 9, 9, 9).permute(0, 3, 1, 2)
        return logits


In [ ]:
from torch import optim
from torch import cuda
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, random_split
import os
import torch
import torch.nn as nn
import pandas as pd

# Dataset names (aliases in DATASET_ALIASES)
DATASET_SPECS = {
    "sudoku-1m": {
        "filenames": ("data/sudoku.csv", "drive/MyDrive/sudoku.csv"),
        "puzzle_col": "quizzes",
        "solution_col": "solutions",
    },
    "sudoku-3m": {
        "filenames": ("data/sudoku-3m.csv", "drive/MyDrive/sudoku-3m.csv"),
        "puzzle_col": "puzzle",
        "solution_col": "solution",
    },
    # From scripts/split_sudoku_3m_by_difficulty.py
    "sudoku-3m-easy": {
        "filenames": ("data/sudoku-3m-easy.csv", "drive/MyDrive/sudoku-3m-easy.csv"),
        "puzzle_col": "puzzle",
        "solution_col": "solution",
    },
    "sudoku-3m-hard": {
        "filenames": ("data/sudoku-3m-hard.csv", "drive/MyDrive/sudoku-3m-hard.csv"),
        "puzzle_col": "puzzle",
        "solution_col": "solution",
    },
}

# Aliases -> canonical dataset names
DATASET_ALIASES = {
    "sudoku": "sudoku-1m",
    "1m": "sudoku-1m",
    "kaggle": "sudoku-1m",
    "3m": "sudoku-3m",
    "easy": "sudoku-3m-easy",
    "hard": "sudoku-3m-hard",
}
TRAIN_DATASET_CHOICES = (
    "sudoku-1m",
    "sudoku-3m",
    "sudoku-3m-easy",
    "sudoku-3m-hard",
)


def resolve_dataset_name(dataset: str) -> str:
    """Resolve dataset alias to canonical name."""
    key = str(dataset).strip()
    key = DATASET_ALIASES.get(key, key)
    if key not in DATASET_SPECS:
        raise ValueError(
            f"Unknown dataset={dataset!r}. Choices: {list(TRAIN_DATASET_CHOICES)}; "
            f"aliases: {list(DATASET_ALIASES.keys())}"
        )
    return key


def normalize_puzzle_string(s: str) -> str:
    """Normalize blank tokens to '0'."""
    return str(s).replace(".", "0")


def sudoku_csv_path(dataset: str = "sudoku-1m") -> str:
    """Return first existing CSV path for dataset."""
    dataset = resolve_dataset_name(dataset)
    for p in DATASET_SPECS[dataset]["filenames"]:
        if os.path.isfile(p):
            return p
    hint = DATASET_SPECS[dataset]["filenames"][0]
    raise FileNotFoundError(
        f"CSV not found for {dataset!r}. Place file at {hint} (Colab: drive/MyDrive/)
    )


def load_sudoku_puzzles_solutions(dataset: str = "sudoku", nrows=None):
    """Load puzzle/solution columns from CSV."""
    path = sudoku_csv_path(dataset)
    spec = DATASET_SPECS[dataset]
    read_kw = {} if nrows is None else {"nrows": nrows}
    df = pd.read_csv(path, **read_kw)
    puzzles = [
        normalize_puzzle_string(str(x)) for x in df[spec["puzzle_col"]].tolist()
    ]
    solutions = [str(x) for x in df[spec["solution_col"]].tolist()]
    return puzzles, solutions


def prepare_supervised_splits(
    dataset: str = "sudoku-1m",
    nrows=None,
    train_ratio: float = 0.8,
    split_seed: int = 42,
):
    """Load CSV and random_split; returns (train_ds, val_ds, name)."""
    name = resolve_dataset_name(dataset)
    puzzles, solutions = load_sudoku_puzzles_solutions(name, nrows=nrows)
    full_ds = SudokuDataset(puzzles, solutions)
    n = len(full_ds)
    n_train = int(train_ratio * n)
    gen = torch.Generator().manual_seed(split_seed)
    train_ds, val_ds = random_split(full_ds, [n_train, n - n_train], generator=gen)
    print(f"[{name}] loaded {n} puzzles -> train {n_train}, val {n - n_train}")
    return train_ds, val_ds, name


class SudokuTrainer:
    """Supervised trainer; loss_on is all_cells or blanks_only."""

    def __init__(
        self,
        model,
        train_dataset,
        val_dataset,
        batch_size=64,
        num_epoch=20,
        save_dir="./checkpoint",
        loss_on: str = "all_cells",
    ):
        if loss_on not in ("all_cells", "blanks_only"):
            raise ValueError(f"loss_on must be all_cells or blanks_only, got {loss_on!r}")
        self.loss_on = loss_on
        self.num_epoch = num_epoch
        self.device = "cuda" if cuda.is_available() else "cpu"
        self.model = model.to(self.device)
        self.optimizer = optim.AdamW(self.model.parameters())
        self.train_dataset, self.val_dataset = train_dataset, val_dataset
        self.train_loader = DataLoader(
            dataset=self.train_dataset, batch_size=batch_size, shuffle=True
        )
        self.val_loader = DataLoader(dataset=self.val_dataset, batch_size=batch_size)
        self.save_dir = save_dir
        # Metrics: blank-cell acc and full-grid exact match
        self.history = {
            "loss_on": loss_on,
            "train_loss": [],
            "train_acc": [],
            "train_puzzle_acc": [],
            "val_loss": [],
            "val_acc": [],
            "val_puzzle_acc": [],
        }

    def _cell_cross_entropy(self, logits, target_classes, puzzles):
        """Cross-entropy over all cells or blanks only."""
        import torch.nn.functional as F

        per_cell = F.cross_entropy(logits, target_classes, reduction="none")
        if self.loss_on == "all_cells":
            return per_cell.mean()
        blank = (puzzles == 0).float()
        denom = blank.sum()
        if denom.item() == 0:
            return per_cell.mean()
        return (per_cell * blank).sum() / denom

    def train_one_epoch(self):
        self.model.train()
        total_loss = 0
        correct_blanks = 0
        total_blanks = 0
        puzzle_correct = 0
        total_puzzles = 0

        pbar = tqdm(self.train_loader, desc=f"Training ({self.loss_on})")
        for puzzles_one_hot, solutions_one_hot in pbar:
            puzzles_one_hot = puzzles_one_hot.to(self.device)
            solutions_one_hot = solutions_one_hot.to(self.device)
            logits = self.model(puzzles_one_hot)
            target_classes = solutions_one_hot.argmax(dim=1).long()
            puzzles = puzzles_one_hot.argmax(dim=1)
            loss = self._cell_cross_entropy(logits, target_classes, puzzles)
            self.optimizer.zero_grad()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

            self.optimizer.step()
            total_loss += loss.item()

            solutions = solutions_one_hot.argmax(dim=1)
            total_blanks += torch.sum(puzzles == 0).item()
            preds = logits.argmax(dim=1)
            correct_blanks += ((preds == solutions) & (puzzles == 0)).sum().item()

            B = preds.size(0)
            total_puzzles += B
            puzzle_correct += (preds == solutions).all(dim=(1, 2)).sum().item()

            blank_acc = correct_blanks / total_blanks if total_blanks > 0 else 0.0
            puzz_acc = puzzle_correct / total_puzzles if total_puzzles > 0 else 0.0
            pbar.set_postfix({
                'loss': loss.item(),
                'cell': blank_acc,
                'puzz': puzz_acc,
            })

        avg_loss = total_loss / len(self.train_loader)
        avg_blank_acc = correct_blanks / total_blanks if total_blanks > 0 else 0.0
        avg_puzzle_acc = puzzle_correct / total_puzzles if total_puzzles > 0 else 0.0

        return avg_loss, avg_blank_acc, avg_puzzle_acc
    
    def validate(self):
        self.model.eval()
        correct_blanks = 0
        total_blanks = 0
        puzzle_correct = 0
        total_puzzles = 0
        total_loss = 0

        with torch.no_grad():

            pbar = tqdm(self.val_loader, desc=f"Validating ({self.loss_on})")
            for puzzles_one_hot, solutions_one_hot in pbar:
                puzzles_one_hot = puzzles_one_hot.to(self.device)
                solutions_one_hot = solutions_one_hot.to(self.device)
                logits = self.model(puzzles_one_hot)
                target_classes = solutions_one_hot.argmax(dim=1).long()
                puzzles = puzzles_one_hot.argmax(dim=1)
                loss = self._cell_cross_entropy(logits, target_classes, puzzles)

                total_loss += loss.item()

                solutions = solutions_one_hot.argmax(dim=1)
                total_blanks += torch.sum(puzzles == 0).item()
                preds = logits.argmax(dim=1)
                correct_blanks += ((preds == solutions) & (puzzles == 0)).sum().item()

                B = preds.size(0)
                total_puzzles += B
                puzzle_correct += (preds == solutions).all(dim=(1, 2)).sum().item()

                blank_acc = correct_blanks / total_blanks if total_blanks > 0 else 0.0
                puzz_acc = puzzle_correct / total_puzzles if total_puzzles > 0 else 0.0
                pbar.set_postfix({
                    'loss': loss.item(),
                    'cell': blank_acc,
                    'puzz': puzz_acc,
                })

        avg_loss = total_loss / len(self.val_loader)
        avg_blank_acc = correct_blanks / total_blanks if total_blanks > 0 else 0.0
        avg_puzzle_acc = puzzle_correct / total_puzzles if total_puzzles > 0 else 0.0

        return avg_loss, avg_blank_acc, avg_puzzle_acc
    
    def train(self):

        best_val_acc = 0
        early_stopping_patience = 5
        patience_counter = 0

        for epoch in range(1, self.num_epoch + 1):
            print(f"\n{'='*50}")
            print(f"Epoch {epoch}/{self.num_epoch}  [loss_on={self.loss_on}]")
            print(f"{'='*50}")

            train_loss, train_cell_acc, train_puzz_acc = self.train_one_epoch()
            val_loss, val_cell_acc, val_puzz_acc = self.validate()

            self.history['train_loss'].append(train_loss)
            self.history['train_acc'].append(train_cell_acc)
            self.history['train_puzzle_acc'].append(train_puzz_acc)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_cell_acc)
            self.history['val_puzzle_acc'].append(val_puzz_acc)

            print(f"\nTrain Loss: {train_loss:.4f}")
            print(
                f"  Train blank-cell acc: {train_cell_acc:.4f}  |  "
                f"Train puzzle-wise acc: {train_puzz_acc:.4f}"
            )
            print(f"Val Loss: {val_loss:.4f}")
            print(
                f"  Val blank-cell acc: {val_cell_acc:.4f}  |  "
                f"Val puzzle-wise acc: {val_puzz_acc:.4f}"
            )

            if val_cell_acc > best_val_acc:
                best_val_acc = val_cell_acc
                # torch.save({
                #     'epoch': epoch,
                #     'model_state_dict': self.model.state_dict(),
                #     'optimizer_state_dict': self.optimizer.state_dict(),
                #     'val_acc': val_cell_acc,
                #     'history': self.history
                # }, os.path.join(self.save_dir, 'best_model.pth'))
                print(
                    f"Save best model (Val blank-cell acc: {val_cell_acc:.4f}, "
                    f"val puzzle-wise: {val_puzz_acc:.4f})"
                )
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= early_stopping_patience:
                    print(f"\nEarly stop. No improvement after {early_stopping_patience} epochs")
                    break

        return self.history


In [ ]:
def SudokuShow(model, puzzle: str, solution: str):
    puzzle_table = SudokuTable(puzzle)
    solution_table = SudokuTable(solution)
    device = next(model.parameters()).device
    puzzle_tensor = puzzle_table.to_tensor().unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(puzzle_tensor)
    answer = logits.argmax(dim=1).squeeze(0)
    # class 0..8 -> digits 1..9
    answer_str = "".join(str(int(x) + 1) for x in answer.flatten().tolist())
    answer_table = SudokuTable(answer_str)
    print("Puzzle:\n")
    print(puzzle_table)
    print("\nPredicted:\n")
    print(answer_table)
    print("\nGround Truth:\n")
    print(solution_table)


In [ ]:
# Failure-case helpers for report figures
import json
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, Tuple, Union

import torch
from torch.utils.data import DataLoader, random_split


def one_hot_puzzle_to_string(puzzle_one_hot: torch.Tensor) -> str:
    """One-hot puzzle tensor -> 81-char string."""
    d = puzzle_one_hot.argmax(dim=0).flatten().tolist()
    return "".join(str(int(x)) for x in d)


def class_grid_to_solution_string(class_grid: torch.Tensor) -> str:
    """Class grid 0..8 -> solution string."""
    return "".join(str(int(x) + 1) for x in class_grid.flatten().tolist())


def build_val_loader_for_capture(
    dataset: str = "sudoku-1m",
    nrows: Optional[int] = 20_000,
    batch_size: int = 64,
    train_ratio: float = 0.8,
    seed: int = 42,
) -> Tuple[DataLoader, List[int]]:
    """Validation DataLoader with same split as *Train."""
    dataset = resolve_dataset_name(dataset)
    puzzles, solutions = load_sudoku_puzzles_solutions(dataset, nrows=nrows)
    full_ds = SudokuDataset(puzzles, solutions)
    n = len(full_ds)
    n_train = int(train_ratio * n)
    g = torch.Generator().manual_seed(seed)
    _tr, val_ds = random_split(full_ds, [n_train, n - n_train], generator=g)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    return val_loader, list(val_ds.indices)


def find_first_supervised_failure(
    model: torch.nn.Module,
    val_loader: DataLoader,
    *,
    device: Optional[torch.device] = None,
    criterion: Literal["blank_cells_only", "any_wrong_cell"] = "blank_cells_only",
    max_batches: Optional[int] = None,
) -> Optional[Dict[str, Any]]:
    """First validation failure (blank_cells_only or any_wrong_cell)."""
    if device is None:
        device = next(model.parameters()).device
    model.eval()
    model = model.to(device)

    for bi, (puzzles_oh, solutions_oh) in enumerate(val_loader):
        if max_batches is not None and bi >= max_batches:
            break
        puzzles_oh = puzzles_oh.to(device)
        solutions_oh = solutions_oh.to(device)
        with torch.no_grad():
            logits = model(puzzles_oh)
        preds = logits.argmax(dim=1)
        targets = solutions_oh.argmax(dim=1)
        clues = puzzles_oh.argmax(dim=1)
        B = preds.size(0)
        for j in range(B):
            wrong = preds[j] != targets[j]
            if criterion == "blank_cells_only":
                blank = clues[j] == 0
                is_fail = bool((wrong & blank).any().item())
            else:
                is_fail = bool(wrong.any().item())
            if not is_fail:
                continue
            puzzle_str = one_hot_puzzle_to_string(puzzles_oh[j].cpu())
            solution_str = class_grid_to_solution_string(targets[j].cpu())
            prediction_str = class_grid_to_solution_string(preds[j].cpu())
            blank_wrong = (wrong & (clues[j] == 0)).sum().item()
            total_wrong = wrong.sum().item()
            wrong_blank_coords: List[Tuple[int, int, int, int]] = []
            for r in range(9):
                for c in range(9):
                    if clues[j, r, c].item() != 0:
                        continue
                    pt, gt = int(preds[j, r, c].item()), int(targets[j, r, c].item())
                    if pt != gt:
                        wrong_blank_coords.append(
                            (r, c, pt + 1, gt + 1)
                        )  # (row, col, pred, gold)
            return {
                "criterion": criterion,
                "batch_index": bi,
                "in_batch_index": j,
                "puzzle": puzzle_str,
                "solution": solution_str,
                "prediction": prediction_str,
                "num_wrong_cells_total": int(total_wrong),
                "num_wrong_blank_cells": int(blank_wrong),
                "wrong_blank_coords": wrong_blank_coords[:40],
            }
    return None


def find_first_ppo_failure(
    agent,
    val_puzzles: List[str],
    val_solutions: List[str],
    device: torch.device,
    *,
    ppo_mode: str = "guided",
    require_full_grid_mismatch: bool = True,
    max_puzzles: Optional[int] = None,
) -> Optional[Dict[str, Any]]:
    """First PPO rollout that does not match the solution."""
    agent.policy_old.load_state_dict(agent.policy.state_dict())
    agent.policy.eval()
    agent.policy_old.eval()

    n = len(val_puzzles) if max_puzzles is None else min(len(val_puzzles), max_puzzles)
    for i in range(n):
        p_str, s_str = val_puzzles[i], val_solutions[i]
        pred_str = rollout_ppo_final_string(agent, p_str, device, ppo_mode=ppo_mode)
        if require_full_grid_mismatch and pred_str == s_str:
            continue
        if not require_full_grid_mismatch:
            # any wrong blank
            bad = any(
                p_str[k] == "0" and pred_str[k] != s_str[k] for k in range(81)
            )
            if not bad:
                continue
        wrong_blank = [
            k
            for k in range(81)
            if p_str[k] == "0" and pred_str[k] != s_str[k]
        ]
        return {
            "puzzle": p_str,
            "solution": s_str,
            "prediction": pred_str,
            "num_wrong_blank_cells": len(wrong_blank),
            "first_wrong_blank_flat_indices": wrong_blank[:40],
            "val_list_index": i,
        }
    return None


def print_failure_for_report(sample: Dict[str, Any], title: str = "") -> None:
    """Print 81-char rows for the report."""
    if sample is None:
        print("(No failure found in scan range.)")
        return
    if title:
        print("=" * 72)
        print(title)
        print("=" * 72)
    for key in ("puzzle", "prediction", "solution"):
        print(f"{key:12s} {sample[key]}")
    for extra in (
        "num_wrong_blank_cells",
        "num_wrong_cells_total",
        "wrong_blank_coords",
        "first_wrong_blank_flat_indices",
    ):
        if extra in sample:
            print(f"{extra}: {sample[extra]}")


def save_failure_json(sample: Dict[str, Any], path: Union[str, Path]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(sample, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"Wrote {path.resolve()}")


# Example usage (uncomment after training)
# val_loader, val_idx = build_val_loader_for_capture("sudoku-1m", nrows=20_000, seed=42)
# ex = find_first_supervised_failure(model, val_loader, criterion="blank_cells_only")
# print_failure_for_report(ex, title="MyModel first failure")
# save_failure_json(ex, "report_assets/mymodel_failure.json")
# Batch over models:
# models = {"MLP": mlp_model, "CNN": cnn_model, "Transformer": tfm_model, "GCN": gcn_model}
# failures = {k: find_first_supervised_failure(m, val_loader) for k, m in models.items()}
# for name, ex in failures.items():
#     print_failure_for_report(ex, title=name)
#     if ex:
#         save_failure_json(ex, f"report_assets/failure_{name.lower()}.json")


In [ ]:
import pandas as pd
from torch.utils.data import random_split


def SudokuMLPTrain(
    dataset: str = "sudoku-1m",
    nrows=None,
    num_epoch: int = 20,
    batch_size: int = 64,
    split_seed: int = 42,
    loss_on: str = "all_cells",
):
    """Train MLP; see DATASET_ALIASES and SudokuTrainer.loss_on."""
    train_ds, val_ds, name = prepare_supervised_splits(
        dataset, nrows=nrows, split_seed=split_seed
    )
    tag = "blanks" if loss_on == "blanks_only" else "allcells"
    model = SudokuMLPModel()
    trainer = SudokuTrainer(
        model,
        train_ds,
        val_ds,
        batch_size=batch_size,
        num_epoch=num_epoch,
        save_dir=f"./checkpoint/mlp_{name}_{tag}",
        loss_on=loss_on,
    )
    return trainer.train()


# Examples:
# SudokuMLPTrain("sudoku-1m")
# SudokuMLPTrain("sudoku-3m-easy", nrows=20_000)
# SudokuMLPTrain("hard")


In [ ]:
from torch.utils.data import random_split


def SudokuCNNTrain(
    dataset: str = "sudoku-1m",
    nrows=None,
    num_epoch: int = 20,
    batch_size: int = 64,
    split_seed: int = 42,
    loss_on: str = "all_cells",
):
    """Train CNN."""
    train_ds, val_ds, name = prepare_supervised_splits(
        dataset, nrows=nrows, split_seed=split_seed
    )
    tag = "blanks" if loss_on == "blanks_only" else "allcells"
    model = SudokuCNNModel()
    trainer = SudokuTrainer(
        model,
        train_ds,
        val_ds,
        batch_size=batch_size,
        num_epoch=num_epoch,
        save_dir=f"./checkpoint/cnn_{name}_{tag}",
        loss_on=loss_on,
    )
    return trainer.train()


# SudokuCNNTrain("sudoku-3m-hard", nrows=20_000)


In [ ]:
import torch
import torch.nn as nn
class SudokuTransformer(nn.Module):
    def __init__(self, d_model, n_head, num_layers):
        super().__init__()
        self.d_model = d_model
        self.n_head = n_head
        self.pos_emb = (nn.Embedding(num_embeddings=81, embedding_dim=d_model))
        self.token_emb = (nn.Embedding(num_embeddings=10, embedding_dim=d_model))
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_head, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer=encoder_layer, num_layers=num_layers)
        self.output_layer = nn.Linear(in_features=d_model, out_features=9)

    def embed_tokens(self, x):
        """Token embeddings before encoder (for attention visualization)."""
        batch_size = x.size(0)
        digits = x.argmax(dim=1).view(batch_size, 81)
        idx = torch.arange(81, device=x.device).unsqueeze(0).expand(batch_size, -1)
        return self.pos_emb(idx) + self.token_emb(digits)

    def forward(self, x):
        batch_size = x.size(0)
        x = self.embed_tokens(x)
        x = self.transformer_encoder(x)
        logits = self.output_layer(x)       # (Batch, 81, 9)
        logits = logits.view(batch_size, 9, 9, 9).permute(0, 3, 1, 2)
        return logits


In [ ]:
import torch
from torch.utils.data import random_split


def SudokuTransformerTrain(
    dataset: str = "sudoku-1m",
    nrows=20_000,
    num_epoch: int = 20,
    batch_size: int = 64,
    split_seed: int = 42,
    loss_on: str = "all_cells",
):
    """Train Transformer; returns model."""
    train_ds, val_ds, name = prepare_supervised_splits(
        dataset, nrows=nrows, split_seed=split_seed
    )
    tag = "blanks" if loss_on == "blanks_only" else "allcells"
    model = SudokuTransformer(128, 4, 4)
    trainer = SudokuTrainer(
        model,
        train_ds,
        val_ds,
        batch_size=batch_size,
        num_epoch=num_epoch,
        save_dir=f"./checkpoint/transformer_{name}_{tag}",
        loss_on=loss_on,
    )
    trainer.train()
    return trainer.model


# tfm_model = SudokuTransformerTrain("sudoku-1m", nrows=20_000, num_epoch=50, loss_on="all_cells")


In [ ]:
from torch.utils.data import random_split


def SudokuGNNTrain(
    dataset: str = "sudoku-1m",
    nrows=20_000,
    num_epoch: int = 20,
    batch_size: int = 64,
    split_seed: int = 42,
    loss_on: str = "all_cells",
):
    """Train GCN; returns model."""
    train_ds, val_ds, name = prepare_supervised_splits(
        dataset, nrows=nrows, split_seed=split_seed
    )
    tag = "blanks" if loss_on == "blanks_only" else "allcells"
    model = SudokuGNNModel(hidden_dim=128, num_layers=6)
    trainer = SudokuTrainer(
        model,
        train_ds,
        val_ds,
        batch_size=batch_size,
        num_epoch=num_epoch,
        save_dir=f"./checkpoint/gnn_{name}_{tag}",
        loss_on=loss_on,
    )
    trainer.train()
    return trainer.model


# gcn_model = SudokuGNNTrain("sudoku-1m", nrows=20_000, num_epoch=50, loss_on="all_cells")


# Supervised loss ablation
SUPERVISED_LOSS_MODES = ("all_cells", "blanks_only")


def compare_supervised_loss_ablation(
    train_fn,
    dataset: str = "sudoku-1m",
    nrows=20_000,
    num_epoch: int = 20,
    batch_size: int = 64,
    split_seed: int = 42,
    **train_kw,
):
    """Run train_fn with both loss_on settings."""
    rows = []
    histories = {}
    for loss_on in SUPERVISED_LOSS_MODES:
        print(f"\n{'='*60}\n[supervised loss ablation] loss_on={loss_on}\n{'='*60}")
        hist = train_fn(
            dataset=dataset,
            nrows=nrows,
            num_epoch=num_epoch,
            batch_size=batch_size,
            split_seed=split_seed,
            loss_on=loss_on,
            **train_kw,
        )
        histories[loss_on] = hist
        rows.append(
            {
                "loss_on": loss_on,
                "val_blank_acc": hist["val_acc"][-1] if hist["val_acc"] else float("nan"),
                "val_puzzle_acc": hist["val_puzzle_acc"][-1]
                if hist["val_puzzle_acc"]
                else float("nan"),
                "val_loss": hist["val_loss"][-1] if hist["val_loss"] else float("nan"),
            }
        )
    print("\n--- Supervised loss ablation (last epoch val) ---")
    print(f"{'loss_on':<14} {'blank-cell':>10} {'puzzle-wise':>12} {'val_loss':>10}")
    for r in rows:
        print(
            f"{r['loss_on']:<14} {r['val_blank_acc']:>10.4f} {r['val_puzzle_acc']:>12.4f} {r['val_loss']:>10.4f}"
        )
    return histories


# compare_supervised_loss_ablation(SudokuTransformerTrain, dataset="sudoku-1m", nrows=20_000, num_epoch=50)


In [ ]:
# Batch supervised runs: 4 models x 3 datasets
import json
from datetime import datetime
from pathlib import Path

import pandas as pd

NROWS = 20_000
NUM_EPOCH = 50
BATCH_SIZE = 64
SPLIT_SEED = 42
LOSS_ON = "blanks_only"  # "blanks_only" | "all_cells"

DATASETS = ("sudoku-1m", "sudoku-3m-easy", "sudoku-3m-hard")

MODEL_REGISTRY = {
    "MLP": SudokuMLPTrain,
    "CNN": SudokuCNNTrain,
    "Transformer": SudokuTransformerTrain,
    "GNN": SudokuGNNTrain,
}

# Debug: MODELS_TO_RUN = ("MLP",)
MODELS_TO_RUN = tuple(MODEL_REGISTRY.keys())

OUT_DIR = Path("report_assets/supervised_batch")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def _hist_last(hist, key):
    return float(hist[key][-1]) if hist.get(key) else float("nan")


def _loss_label(loss_on: str) -> str:
    return "all cells" if loss_on == "all_cells" else "blanks only"


def run_supervised_batch(
    models=MODELS_TO_RUN,
    datasets=DATASETS,
    loss_on=LOSS_ON,
    nrows=NROWS,
    num_epoch=NUM_EPOCH,
    batch_size=BATCH_SIZE,
    split_seed=SPLIT_SEED,
    out_dir=OUT_DIR,
):
    """Train all model/dataset pairs."""
    rows = []
    total = len(models) * len(datasets)
    run_idx = 0

    for model_name in models:
        train_fn = MODEL_REGISTRY[model_name]
        for dataset in datasets:
            ds_name = resolve_dataset_name(dataset)
            run_idx += 1
            print(
                f"\n{'='*72}\n"
                f"[{run_idx}/{total}] {model_name}\n"
                f"  dataset size: {nrows}, {ds_name}, {num_epoch} epoch, "
                f"loss on {_loss_label(loss_on)}\n"
                f"{'='*72}"
            )
            t0 = datetime.now()
            try:
                hist = train_fn(
                    dataset=dataset,
                    nrows=nrows,
                    num_epoch=num_epoch,
                    batch_size=batch_size,
                    split_seed=split_seed,
                    loss_on=loss_on,
                )
                row = {
                    "model": model_name,
                    "dataset": ds_name,
                    "nrows": nrows,
                    "num_epoch": num_epoch,
                    "loss_on": loss_on,
                    "val_blank_acc": _hist_last(hist, "val_acc"),
                    "val_puzzle_acc": _hist_last(hist, "val_puzzle_acc"),
                    "val_loss": _hist_last(hist, "val_loss"),
                    "train_blank_acc": _hist_last(hist, "train_acc"),
                    "train_puzzle_acc": _hist_last(hist, "train_puzzle_acc"),
                    "status": "ok",
                    "error": "",
                    "elapsed_sec": (datetime.now() - t0).total_seconds(),
                }
            except Exception as e:
                row = {
                    "model": model_name,
                    "dataset": ds_name,
                    "nrows": nrows,
                    "num_epoch": num_epoch,
                    "loss_on": loss_on,
                    "val_blank_acc": float("nan"),
                    "val_puzzle_acc": float("nan"),
                    "val_loss": float("nan"),
                    "train_blank_acc": float("nan"),
                    "train_puzzle_acc": float("nan"),
                    "status": "failed",
                    "error": str(e),
                    "elapsed_sec": (datetime.now() - t0).total_seconds(),
                }
                print(f"FAILED: {e}")

            rows.append(row)
            print(
                f"  Val blank-cell acc: {row['val_blank_acc']:.4f}  |  "
                f"Val puzzle-wise acc: {row['val_puzzle_acc']:.4f}  "
                f"({row['elapsed_sec']:.0f}s)"
            )

    df = pd.DataFrame(rows)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    tag = "blanks" if loss_on == "blanks_only" else "allcells"
    csv_path = out_dir / f"supervised_batch_{tag}_{stamp}.csv"
    json_path = out_dir / f"supervised_batch_{tag}_{stamp}.json"
    df.to_csv(csv_path, index=False)
    json_path.write_text(
        json.dumps(rows, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(f"\nSaved:\n  {csv_path}\n  {json_path}")

    print("\n" + "=" * 72)
    print("SUMMARY (last epoch validation)")
    print("=" * 72)
    _train_fn_names = {
        "MLP": "SudokuMLPTrain",
        "CNN": "SudokuCNNTrain",
        "Transformer": "Transformer",
        "GNN": "GNN",
    }
    for model_name in models:
        sub = df[df["model"] == model_name]
        if sub.empty:
            continue
        print(f"\n{_train_fn_names.get(model_name, model_name)}")
        for _, r in sub.iterrows():
            print(
                f"dataset size: {int(r['nrows'])}, {r['dataset']}, "
                f"{int(r['num_epoch'])} epoch, loss on {_loss_label(r['loss_on'])}\n"
                f"Val blank-cell acc: {r['val_blank_acc']:.4f}  |  "
                f"Val puzzle-wise acc: {r['val_puzzle_acc']:.4f}\n"
            )

    return df


# Uncomment to run full batch (long)
# supervised_results_df = run_supervised_batch()

# run_supervised_batch(loss_on="all_cells")


In [ ]:
import torch
import torch.nn as nn


class PolicyNetwork(nn.Module):
    """Example policy network."""

    def __init__(self, n_head, num_layers, num_actions=729, d_model=128):
        super().__init__()
        self.d_model = d_model
        self.n_head = n_head
        self.num_layers = num_layers

        self.pos_emb = nn.Embedding(num_embeddings=81, embedding_dim=d_model)
        self.token_emb = nn.Embedding(num_embeddings=10, embedding_dim=d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_head, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer, num_layers=num_layers
        )

        self.position_head = nn.Linear(in_features=d_model, out_features=81)
        self.value_head = nn.Linear(in_features=d_model, out_features=9)

    def forward(self, x):
        if x.dim() == 4 and x.size(1) == 10:
            batch_size = x.size(0)
            device = x.device
            digits = x.argmax(dim=1).view(batch_size, 81)
        elif x.dim() == 2 and x.size(1) == 81:
            batch_size = x.size(0)
            device = x.device
            digits = x.long()
        else:
            raise ValueError(
                f"PolicyNetwork: expected (B,10,9,9) or (B,81), got {tuple(x.shape)}"
            )

        idx = torch.arange(81, device=device).unsqueeze(0).expand(batch_size, -1)
        h = self.token_emb(digits) + self.pos_emb(idx)
        cls = self.cls_token.expand(batch_size, -1, -1)
        h = torch.cat([cls, h], dim=1)
        h = self.transformer_encoder(h)
        cls_out = h[:, 0]
        position_logits = self.position_head(cls_out)
        value_logits = self.value_head(cls_out)
        return position_logits, value_logits


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
from copy import deepcopy
import numpy as np

# ==========================================
# Actor-critic Transformer
# ==========================================
class SudokuPPOModel(nn.Module):
    def __init__(self, d_model=128, n_head=4, num_layers=4):
        super().__init__()
        self.d_model = d_model
        # Embeddings
        self.pos_emb = nn.Embedding(81, d_model)
        self.token_emb = nn.Embedding(11, d_model) 
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_head, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Actor heads
        self.pos_head = nn.Linear(d_model, 1)
        self.digit_head = nn.Linear(d_model, 9)

        # Critic
        self.critic_head = nn.Sequential(
            nn.Linear(d_model * 81, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, x, mask=None):
        # x: (B, 81) token ids
        batch_size = x.size(0)
        device = x.device
        idx = torch.arange(81, device=device).unsqueeze(0).expand(batch_size, -1)
        
        # token + position embeddings
        h = self.token_emb(x) + self.pos_emb(idx)
        h_encoded = self.transformer(h) # (Batch, 81, d_model)

        # Actor
        # position logits
        pos_logits = self.pos_head(h_encoded).squeeze(-1) # (Batch, 81)
        if mask is not None:
            # mask shape: (Batch, 81) where True means valid positions
            pos_logits = pos_logits.masked_fill(~mask.bool(), -1e9)
        
        # digit logits
        digit_logits = self.digit_head(h_encoded) # (Batch, 81, 9)

        # Critic
        state_value = self.critic_head(h_encoded.view(batch_size, -1)) # (Batch, 1)

        return pos_logits, digit_logits, state_value


In [ ]:
# ==========================================
# PPO agent
# ==========================================

# PPO 2x2: MRV x candidate mask
PPO_MODES_2X2 = ("guided", "mrv_only", "mask_only", "free")

_PPO_ACTION_TABLE = {
    # MRV + candidate mask
    "guided": {"use_mrv_position": True, "use_valid_candidates": True},
    "mrv_mask": {"use_mrv_position": True, "use_valid_candidates": True},
    "constrained": {"use_mrv_position": True, "use_valid_candidates": True},
    # MRV; 9-way digit policy
    "mrv_only": {"use_mrv_position": True, "use_valid_candidates": False},
    "mrv": {"use_mrv_position": True, "use_valid_candidates": False},
    # policy position; masked digits
    "mask_only": {"use_mrv_position": False, "use_valid_candidates": True},
    "mask": {"use_mrv_position": False, "use_valid_candidates": True},
    "candidate_mask": {"use_mrv_position": False, "use_valid_candidates": True},
    # no heuristics
    "free": {"use_mrv_position": False, "use_valid_candidates": False},
    "none": {"use_mrv_position": False, "use_valid_candidates": False},
    "unconstrained": {"use_mrv_position": False, "use_valid_candidates": False},
}


def get_ppo_action_config(mode: str) -> dict:
    """PPO action config for guided | mrv_only | mask_only | free."""
    key = str(mode).lower().strip()
    if key not in _PPO_ACTION_TABLE:
        raise ValueError(
            f"Unknown ppo_mode={mode!r}. Choices: {list(PPO_MODES_2X2)}; "
            f"see _PPO_ACTION_TABLE.keys()"
        )
    return dict(_PPO_ACTION_TABLE[key])


def ppo_mode_label(mode: str) -> str:
    """Short label for PPO mode."""
    cfg = get_ppo_action_config(mode)
    mrv = "MRV" if cfg["use_mrv_position"] else "policy pos"
    mask = "cand.mask" if cfg["use_valid_candidates"] else "9-way digit"
    return f"{mode} ({mrv} | {mask})"


class PPOAgent:
    def __init__(self, model, lr=1e-3, gamma=0.99, eps_clip=0.3, K_epochs=10):
        self.gamma = gamma
        self.eps_clip = eps_clip
        self.K_epochs = K_epochs
        self.device = next(model.parameters()).device
        
        self.policy = model
        self.optimizer = torch.optim.AdamW(self.policy.parameters(), lr=lr)
        self.policy_old = deepcopy(model)
        self.MseLoss = nn.MSELoss()

    def select_action(
        self,
        state,
        env,
        use_mrv_position=False,
        use_valid_candidates=False,
        greedy=None,
    ):
        """Sample (position, digit) with optional MRV and candidate masking."""
        if greedy is not None:
            use_mrv_position = greedy

        state = state.to(self.device)

        with torch.no_grad():
            mask = (state == 0).float()
            if mask.sum() == 0:
                return -1, -1, None, None, mask

            pos_logits, digit_logits, state_value = self.policy_old(state, mask=mask)

            # position
            if use_mrv_position:
                row_col_dofs = []
                for k in range(81):
                    if mask[0, k] == 1:
                        r, c = k // 9, k % 9
                        dof = len(env.candidates(r, c))
                        row_col_dofs.append((dof, k, r, c))
                if not row_col_dofs:
                    return -1, -1, None, state_value, mask
                _, pos_action_val, row, col = min(row_col_dofs)
                pos_action = torch.tensor(pos_action_val, device=self.device)
            else:
                # sample over empty cells only
                blank_idx = torch.nonzero(mask[0] > 0, as_tuple=False).squeeze(-1)
                if blank_idx.numel() == 0:
                    return -1, -1, None, state_value, mask
                blank_logits = pos_logits[0, blank_idx]
                blank_probs = F.softmax(blank_logits, dim=-1)
                local_pos = Categorical(blank_probs).sample()
                pos_action = blank_idx[local_pos]
                row, col = pos_action.item() // 9, pos_action.item() % 9
                pos_log_prob = torch.log(blank_probs[local_pos] + 1e-8)

            # digit
            digit_logits_pos = digit_logits[0, pos_action]
            digit_probs = F.softmax(digit_logits_pos, dim=-1)

            if use_valid_candidates:
                candidates = env.candidates(row, col)
                if len(candidates) == 0:
                    return pos_action.item(), -1, None, state_value, mask
                candidate_mask = torch.zeros(9, device=self.device)
                for d in candidates:
                    candidate_mask[d - 1] = 1.0
                digit_probs = digit_probs * candidate_mask
                prob_sum = digit_probs.sum()
                if prob_sum.item() <= 1e-8:
                    digit_probs = candidate_mask / (candidate_mask.sum() + 1e-8)
                else:
                    digit_probs = digit_probs / prob_sum

            dist_digit = Categorical(digit_probs)
            digit_action = dist_digit.sample()

            if use_mrv_position:
                log_prob = torch.log(digit_probs[digit_action] + 1e-8)
            else:
                log_prob = pos_log_prob + torch.log(digit_probs[digit_action] + 1e-8)

        return pos_action.item(), digit_action.item(), log_prob, state_value, mask

    def update(self, memory):
        states = torch.stack(memory.states).to(self.device)
        masks = torch.stack(memory.masks).to(self.device)
        actions_pos = torch.tensor(memory.actions_pos, device=self.device, dtype=torch.long)
        actions_digit = torch.tensor(memory.actions_digit, device=self.device, dtype=torch.long)
        old_log_probs = torch.stack(memory.log_probs).to(self.device)
        rewards = torch.tensor(memory.rewards, device=self.device, dtype=torch.float32)
        is_terminals = torch.tensor(memory.is_terminals, device=self.device, dtype=torch.bool)
        
        returns = []
        discounted_reward = 0
        for reward, is_terminal in zip(reversed(memory.rewards), reversed(memory.is_terminals)):
            if is_terminal: 
                discounted_reward = 0
            discounted_reward = reward + (self.gamma * discounted_reward)
            returns.insert(0, discounted_reward)
        returns = torch.tensor(returns, device=self.device, dtype=torch.float32)

        for _ in range(self.K_epochs):
            pos_logits, digit_logits, state_values = self.policy(states, mask=masks)
            
            pos_probs = F.softmax(pos_logits, dim=-1)
            dist_pos = Categorical(pos_probs)
            batch_size = states.size(0)
            batch_indices = torch.arange(batch_size, device=self.device)
            selected_digit_logits = digit_logits[batch_indices, actions_pos]
            digit_probs = F.softmax(selected_digit_logits, dim=-1)
            dist_digit = Categorical(digit_probs)
            
            new_log_probs = dist_pos.log_prob(actions_pos) + dist_digit.log_prob(actions_digit)
            entropy = (dist_pos.entropy() + dist_digit.entropy()).mean()
            
            ratios = torch.exp(new_log_probs - old_log_probs)
            advantages = returns - state_values.squeeze(-1)
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
            
            surr1 = ratios * advantages
            surr2 = torch.clamp(ratios, 1-self.eps_clip, 1+self.eps_clip) * advantages
            
            policy_loss = -torch.min(surr1, surr2).mean()
            value_loss = 0.5 * self.MseLoss(state_values.squeeze(-1), returns)
            entropy_bonus = -0.01 * entropy
            loss = policy_loss + value_loss + entropy_bonus
            
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.policy.parameters(), max_norm=0.5)
            self.optimizer.step()
            
        self.policy_old.load_state_dict(self.policy.state_dict())


In [ ]:
# ==========================================
# Memory and training loop
# ==========================================
class Memory:
    def __init__(self):
        self.states, self.masks, self.actions_pos, self.actions_digit, self.log_probs, self.rewards, self.is_terminals = [], [], [], [], [], [], []
    
    def clear(self):
        del self.states[:], self.masks[:], self.actions_pos[:], self.actions_digit[:], self.log_probs[:], self.rewards[:], self.is_terminals[:]

def train_ppo_sudoku(
    puzzles,
    solutions,
    num_episodes=1000,
    device="cpu",
    ppo_mode: str = "guided",
    greedy_policy=None,
):
    """PPO training loop (guided | mrv_only | mask_only | free)."""
    action_cfg = get_ppo_action_config(ppo_mode)
    if greedy_policy is not None:
        action_cfg["use_mrv_position"] = greedy_policy
    print(f"[train_ppo_sudoku] ppo_mode={ppo_mode!r} -> {action_cfg}")

    env_model = SudokuPPOModel().to(device)
    agent = PPOAgent(env_model)
    memory = Memory()

    episode_rewards = []
    solve_count = 0
    illegal_count = 0

    for eps in range(num_episodes):
        raw_str = puzzles[np.random.randint(len(puzzles))]
        env = SudokuTable(raw_str)
        state = torch.tensor(
            [int(c) for c in env.to_string()], dtype=torch.long
        ).unsqueeze(0).to(device)

        episode_reward = 0
        for t in range(81):
            pos_idx, digit_val, log_p, state_val, mask = agent.select_action(
                state, env, **action_cfg
            )
            
            if pos_idx == -1 or digit_val == -1:
                break
            
            _, reward, done, info = env.step(pos_idx // 9, pos_idx % 9, digit_val + 1)
            
            # shaped reward
            if info.get("valid"):
                reward = 0.2
            else:
                reward = -0.2
                illegal_count += 1
            
            memory.states.append(state.squeeze().to('cpu'))
            memory.masks.append(mask.squeeze().to('cpu'))
            memory.actions_pos.append(pos_idx)
            memory.actions_digit.append(digit_val)
            memory.log_probs.append(log_p.to('cpu'))
            memory.rewards.append(reward)
            memory.is_terminals.append(done)
            
            episode_reward += reward
            
            if done:
                if info.get('solved'):
                    solve_count += 1
                break
            
            state = torch.tensor([int(c) for c in env.to_string()], dtype=torch.long).unsqueeze(0).to(device)
            
        episode_rewards.append(episode_reward)
        
        if (eps + 1) % 10 == 0:
            if len(memory.states) > 0:
                agent.update(memory)
                memory.clear()
            
        if (eps + 1) % 50 == 0:
            avg_reward = np.mean(episode_rewards[-50:])
            print(
                f"Episode {eps+1}/{num_episodes}, Avg Reward: {avg_reward:.4f}, "
                f"Solved: {solve_count}/{eps + 1}, illegal steps (cumul.): {illegal_count}"
            )

    return agent, episode_rewards


def train_ppo_2x2_ablation(
    puzzles,
    solutions,
    modes=None,
    num_episodes=500,
    device=None,
    dataset_tag="sudoku-1m",
    save_checkpoints=True,
):
    """Train all PPO 2x2 modes."""
    if modes is None:
        modes = PPO_MODES_2X2
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    ds_name = resolve_dataset_name(dataset_tag)
    agents = {}
    for mode in modes:
        print(f"\n{'#'*60}\n# PPO 2×2 train: {ppo_mode_label(mode)}\n{'#'*60}")
        agent, rewards = train_ppo_sudoku(
            puzzles, solutions, num_episodes=num_episodes, device=device, ppo_mode=mode
        )
        agents[mode] = (agent, rewards)
        if save_checkpoints:
            ckpt = f"ppo_{mode}_{ds_name}.pth"
            torch.save(agent.policy.state_dict(), ckpt)
            print(f"Saved {ckpt}")
    return agents



In [ ]:
# ==========================================
# PPO 2x2 entry (run eval cell first)
# ==========================================
# PPO 2x2 grid: MRV/mask -> guided/mrv_only/mask_only/free
pass


In [ ]:
# ==========================================
# Optional: train a single PPO mode
# ==========================================
# train_ppo_sudoku(puzzles, solutions, num_episodes=500, device=device, ppo_mode="mrv_only")
# train_ppo_sudoku(puzzles, solutions, num_episodes=500, device=device, ppo_mode="mask_only")

# Load checkpoint for eval only:
# def load_ppo_agent(ckpt_path, device):
#     m = SudokuPPOModel().to(device)
#     m.load_state_dict(torch.load(ckpt_path, map_location=device))
#     return PPOAgent(m)
# ppo_agents_2x2 = {
#     "guided": (load_ppo_agent(f"ppo_guided_{resolve_dataset_name('sudoku-1m')}.pth", device), []),
#     ...
# }
# compare_ppo_2x2_table(ppo_agents_2x2, puzzles, solutions, device)


In [ ]:
# ==========================================
# Evaluate trained PPO
# ==========================================
from torch.utils.data import random_split


def make_supervised_style_val_lists(
    puzzles=None,
    solutions=None,
    *,
    dataset: str | None = None,
    nrows=None,
    train_ratio=0.8,
    seed=42,
):
    """Validation puzzle/solution lists (same split as *Train)."""
    if dataset is not None:
        name = resolve_dataset_name(dataset)
        puzzles, solutions = load_sudoku_puzzles_solutions(name, nrows=nrows)
    if puzzles is None or solutions is None:
        raise ValueError("Need (puzzles, solutions) or dataset=...")
    full_ds = SudokuDataset(puzzles, solutions)
    n = len(full_ds)
    n_train = int(train_ratio * n)
    g = torch.Generator().manual_seed(seed)
    _train_ds, val_ds = random_split(full_ds, [n_train, n - n_train], generator=g)
    idx = val_ds.indices
    return [puzzles[i] for i in idx], [solutions[i] for i in idx]


def rollout_ppo_final_string(
    agent,
    puzzle_str,
    device,
    ppo_mode: str = "guided",
    greedy=None,
    use_valid_candidates=None,
    max_steps=81,
    **action_overrides,
):
    """Roll out PPO to a final 81-char board string."""
    action_cfg = get_ppo_action_config(ppo_mode)
    if greedy is not None:
        action_cfg["use_mrv_position"] = greedy
    if use_valid_candidates is not None:
        action_cfg["use_valid_candidates"] = use_valid_candidates
    action_cfg.update(action_overrides)

    env = SudokuTable(puzzle_str)
    state = torch.tensor(
        [int(c) for c in env.to_string()], dtype=torch.long
    ).unsqueeze(0).to(device)
    for _ in range(max_steps):
        pos_idx, digit_val, _, _, _ = agent.select_action(state, env, **action_cfg)
        if pos_idx == -1 or digit_val == -1:
            break
        _, _, done, _ = env.step(pos_idx // 9, pos_idx % 9, digit_val + 1)
        if done:
            break
        state = torch.tensor(
            [int(c) for c in env.to_string()], dtype=torch.long
        ).unsqueeze(0).to(device)
    return env.to_string()


def evaluate_ppo_blank_cell_accuracy(
    agent,
    puzzles,
    solutions,
    device,
    ppo_mode: str = "guided",
    sync_policy_old=True,
    show_progress=True,
    **action_overrides,
):
    """Blank-cell and full-grid accuracy (same metrics as supervised)."""
    if sync_policy_old:
        agent.policy_old.load_state_dict(agent.policy.state_dict())
    agent.policy.eval()
    agent.policy_old.eval()

    total_blanks = 0
    correct_blanks = 0
    full_exact = 0
    if show_progress:
        from tqdm import tqdm

        iterator = tqdm(
            zip(puzzles, solutions),
            total=len(puzzles),
            desc="PPO eval (supervised-style)",
        )
    else:
        iterator = zip(puzzles, solutions)

    for p_str, s_str in iterator:
        pred_str = rollout_ppo_final_string(
            agent,
            p_str,
            device,
            ppo_mode=ppo_mode,
            **action_overrides,
        )
        if pred_str == s_str:
            full_exact += 1
        for i in range(81):
            if p_str[i] == "0":
                total_blanks += 1
                if pred_str[i] == s_str[i]:
                    correct_blanks += 1

    blank_acc = correct_blanks / total_blanks if total_blanks > 0 else 0.0
    full_acc = full_exact / len(puzzles) if puzzles else 0.0
    return blank_acc, full_acc


def compare_ppo_2x2_table(agents_by_mode, val_puzzles, val_solutions, device, modes=None):
    """Evaluate each PPO mode on validation set."""
    if modes is None:
        modes = PPO_MODES_2X2
    print(f"\n--- PPO 2×2 validation (n={len(val_puzzles)}) ---")
    print(f"{'mode':<12} {'MRV':^5} {'mask':^5} {'blank-cell':>10} {'puzzle-wise':>12}")
    print("-" * 56)
    rows = []
    for mode in modes:
        if mode not in agents_by_mode:
            continue
        agent = agents_by_mode[mode]
        if isinstance(agent, tuple):
            agent = agent[0]
        cfg = get_ppo_action_config(mode)
        blank_acc, puzzle_acc = evaluate_ppo_blank_cell_accuracy(
            agent, val_puzzles, val_solutions, device, ppo_mode=mode, show_progress=False
        )
        rows.append(
            {
                "mode": mode,
                "val_blank_acc": blank_acc,
                "val_puzzle_acc": puzzle_acc,
                **cfg,
            }
        )
        print(
            f"{mode:<12} {str(cfg['use_mrv_position']):^5} "
            f"{str(cfg['use_valid_candidates']):^5} "
            f"{blank_acc:>10.4f} {puzzle_acc:>12.4f}"
        )
    return rows


def train_and_eval_ppo_2x2(
    dataset: str = "sudoku-1m",
    nrows=20_000,
    num_episodes=500,
    train_ratio=0.8,
    split_seed=42,
    modes=None,
    device=None,
    save_checkpoints=True,
    out_dir=None,
):
    """Train and evaluate all PPO 2x2 modes."""
    import json
    from datetime import datetime
    from pathlib import Path

    import pandas as pd

    if modes is None:
        modes = PPO_MODES_2X2
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    out_dir = Path(out_dir or "report_assets/ppo_2x2")
    out_dir.mkdir(parents=True, exist_ok=True)

    ds_name = resolve_dataset_name(dataset)
    puzzles, solutions = load_sudoku_puzzles_solutions(dataset, nrows=nrows)
    val_p, val_s = make_supervised_style_val_lists(
        puzzles, solutions, train_ratio=train_ratio, seed=split_seed
    )
    print(
        f"PPO 2×2 | {ds_name} | train n={len(puzzles)} | val n={len(val_p)} | "
        f"episodes={num_episodes} | device={device}"
    )

    agents, rows = {}, []
    for i, mode in enumerate(modes, 1):
        print(f"\n{'#'*72}\n# [{i}/{len(modes)}] TRAIN  {ppo_mode_label(mode)}\n{'#'*72}")
        agent, rewards = train_ppo_sudoku(
            puzzles, solutions, num_episodes=num_episodes, device=device, ppo_mode=mode
        )
        agents[mode] = (agent, rewards)
        if save_checkpoints:
            ckpt = out_dir / f"ppo_{mode}_{ds_name}.pth"
            torch.save(agent.policy.state_dict(), ckpt)
            print(f"Saved {ckpt}")

        print(f"\n--- EVAL  {mode} ---")
        blank_acc, puzzle_acc = evaluate_ppo_blank_cell_accuracy(
            agent, val_p, val_s, device, ppo_mode=mode, show_progress=True
        )
        print(
            f"Val blank-cell acc: {blank_acc:.4f}  |  Val puzzle-wise acc: {puzzle_acc:.4f}"
        )
        cfg = get_ppo_action_config(mode)
        rows.append(
            {
                "mode": mode,
                "dataset": ds_name,
                "nrows": nrows,
                "num_episodes": num_episodes,
                "use_mrv": cfg["use_mrv_position"],
                "use_candidate_mask": cfg["use_valid_candidates"],
                "val_blank_acc": blank_acc,
                "val_puzzle_acc": puzzle_acc,
            }
        )

    df = pd.DataFrame(rows)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = out_dir / f"ppo_2x2_{ds_name}_{stamp}.csv"
    df.to_csv(csv_path, index=False)
    (out_dir / f"ppo_2x2_{ds_name}_{stamp}.json").write_text(
        json.dumps(rows, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(f"\nSaved {csv_path}")
    compare_ppo_2x2_table(agents, val_p, val_s, device, modes=modes)
    return df, agents


# One-click run (functions above must be defined)
import matplotlib.pyplot as plt

TRAIN_DATASET = "sudoku-1m"
PPO_NROWS = 20_000
NUM_EPISODES = 500
SPLIT_SEED = 42
PPO_MODES_TO_RUN = PPO_MODES_2X2

# Uncomment: train 4 modes + validation metrics
# ppo_2x2_results_df, ppo_agents_2x2 = train_and_eval_ppo_2x2(
#     dataset=TRAIN_DATASET,
#     nrows=PPO_NROWS,
#     num_episodes=NUM_EPISODES,
#     split_seed=SPLIT_SEED,
#     modes=PPO_MODES_TO_RUN,
# )

# Plot curves after training:
# plt.figure(figsize=(10, 4))
# for mode, (_ag, rew) in ppo_agents_2x2.items():
#     if len(rew) >= 50:
#         ma = np.convolve(rew, np.ones(50) / 50, mode="valid")
#         plt.plot(range(49, 49 + len(ma)), ma, label=mode, linewidth=1.5)
# plt.xlabel("Episode"); plt.ylabel("Reward MA(50)"); plt.title("PPO 2×2"); plt.legend(); plt.grid(True); plt.show()


def test_ppo_agent(agent, puzzle_str, device="cpu", ppo_mode: str = "guided"):
    """Solve one puzzle with PPO (debug)."""
    env = SudokuTable(puzzle_str)
    state = torch.tensor([int(c) for c in env.to_string()], dtype=torch.long).unsqueeze(0).to(device)

    print("Initial board:")
    print(env)

    is_solved = False
    steps = 0

    for t in range(81):
        pos_idx, digit_val, _, _, _ = agent.select_action(
            state, env, **get_ppo_action_config(ppo_mode)
        )
        if pos_idx == -1 or digit_val == -1:
            break
        _, reward, done, info = env.step(pos_idx // 9, pos_idx % 9, digit_val + 1)

        steps += 1

        if done:
            if info.get("solved"):
                is_solved = True
            break

        state = torch.tensor([int(c) for c in env.to_string()], dtype=torch.long).unsqueeze(0).to(device)
    
    print(f"\nBoard after {steps} steps:")
    print(env)
    print(f"Solved: {is_solved}")
    print(f"Valid: {env.is_valid()}")
    print(f"Complete: {env.is_complete()}")
    
    return is_solved, env

# Demo on random puzzles
if "agent" not in globals():
    print("Train PPO in the previous cell first.")
else:
    num_test = 5
    test_indices = np.random.choice(len(puzzles), num_test, replace=False)

    solve_count = 0
    for k, idx in enumerate(test_indices):
        print("\n" + "=" * 60)
        print(f"Test {k + 1}/{num_test} (puzzle index {idx})")
        print("=" * 60)
        is_solved, env = test_ppo_agent(agent, puzzles[idx], device=device)
        if is_solved:
            solve_count += 1

    print(f"\nSolved: {solve_count}/{num_test}")
    print(f"Success rate: {solve_count / num_test * 100:.1f}%")

    # Same validation split as supervised
    val_p, val_s = make_supervised_style_val_lists(
        puzzles, solutions, train_ratio=0.8, seed=42
    )
    # Subsample val if needed: val_p[:500]
    blank_acc, full_acc = evaluate_ppo_blank_cell_accuracy(
        agent,
        val_p,
        val_s,
        device,
        ppo_mode="guided",
    )
    print(
        f"\n[validation] blank-cell accuracy = {blank_acc:.4f} | "
        f"full-grid exact match = {full_acc:.4f}"
    )


In [ ]:
# ==========================================
# PPO 2x2 one-click (run cell above first)
# ==========================================
ppo_2x2_results_df, ppo_agents_2x2 = train_and_eval_ppo_2x2(
    dataset="sudoku-1m",
    nrows=20_000,
    num_episodes=500,
    split_seed=42,
    modes=PPO_MODES_2X2,
)

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
for mode, (_ag, rew) in ppo_agents_2x2.items():
    if len(rew) >= 50:
        ma = np.convolve(rew, np.ones(50) / 50, mode="valid")
        plt.plot(range(49, 49 + len(ma)), ma, label=mode, linewidth=1.5)
plt.xlabel("Episode")
plt.ylabel("Reward MA(50)")
plt.title("PPO 2×2 training")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

display(ppo_2x2_results_df)


In [ ]:
# ==========================================
# PPO 2x2 train + test evaluation
# ==========================================
# Requires SudokuTable, SudokuPPOModel, PPOAgent, get_ppo_action_config.
# Reports blank-cell and full-grid accuracy per mode.

from datetime import datetime
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

PPO_DATASET = "sudoku-1m"
PPO_NROWS = 20_000
PPO_NUM_EPISODES = 500
PPO_SPLIT_SEED = 42
PPO_TRAIN_RATIO = 0.8
PPO_PRINT_EVERY = 5
PPO_EVAL_LIMIT = None
PPO_MODES_TO_RUN = ("guided", "mrv_only", "mask_only", "free")


def train_ppo_sudoku_progress(
    puzzles,
    solutions,
    num_episodes=500,
    device=None,
    ppo_mode="guided",
    print_every=5,
):
    """PPO training with frequent progress logs."""
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    action_cfg = get_ppo_action_config(ppo_mode)
    print(f"[train_ppo_sudoku_progress] ppo_mode={ppo_mode!r} -> {action_cfg}")

    env_model = SudokuPPOModel().to(device)
    agent = PPOAgent(env_model)
    memory = Memory()

    episode_rewards = []
    solve_count = 0
    illegal_count = 0

    for eps in range(num_episodes):
        raw_str = puzzles[np.random.randint(len(puzzles))]
        env = SudokuTable(raw_str)
        state = torch.tensor([int(c) for c in env.to_string()], dtype=torch.long).unsqueeze(0).to(device)

        episode_reward = 0
        step_count = 0
        for _ in range(81):
            pos_idx, digit_val, log_p, _state_val, mask = agent.select_action(state, env, **action_cfg)
            if pos_idx == -1 or digit_val == -1:
                break

            _, reward, done, info = env.step(pos_idx // 9, pos_idx % 9, digit_val + 1)
            if info.get("valid"):
                reward = 0.2
            else:
                reward = -0.2
                illegal_count += 1

            memory.states.append(state.squeeze().to("cpu"))
            memory.masks.append(mask.squeeze().to("cpu"))
            memory.actions_pos.append(pos_idx)
            memory.actions_digit.append(digit_val)
            memory.log_probs.append(log_p.to("cpu"))
            memory.rewards.append(reward)
            memory.is_terminals.append(done)

            episode_reward += reward
            step_count += 1
            if done:
                if info.get("solved"):
                    solve_count += 1
                break

            state = torch.tensor([int(c) for c in env.to_string()], dtype=torch.long).unsqueeze(0).to(device)

        episode_rewards.append(episode_reward)

        if (eps + 1) % 10 == 0 and len(memory.states) > 0:
            agent.update(memory)
            memory.clear()

        if (eps + 1) % print_every == 0 or eps == 0:
            recent = episode_rewards[-print_every:] if len(episode_rewards) >= print_every else episode_rewards
            avg_reward = float(np.mean(recent))
            print(
                f"Episode {eps + 1}/{num_episodes} | avg reward({len(recent)})={avg_reward:.4f} | "
                f"last steps={step_count} | solved={solve_count}/{eps + 1} | illegal={illegal_count}"
            )

    return agent, episode_rewards


def evaluate_ppo_cell_and_puzzle_accuracy(
    agent,
    test_puzzles,
    test_solutions,
    device,
    ppo_mode="guided",
    eval_limit=None,
    show_progress=True,
):
    """Return (blank-cell acc, full-grid acc)."""
    if eval_limit is not None:
        test_puzzles = test_puzzles[:eval_limit]
        test_solutions = test_solutions[:eval_limit]
    return evaluate_ppo_blank_cell_accuracy(
        agent,
        test_puzzles,
        test_solutions,
        device,
        ppo_mode=ppo_mode,
        show_progress=show_progress,
    )


def run_ppo_2x2_train_and_test(
    dataset=PPO_DATASET,
    nrows=PPO_NROWS,
    num_episodes=PPO_NUM_EPISODES,
    modes=PPO_MODES_TO_RUN,
    train_ratio=PPO_TRAIN_RATIO,
    split_seed=PPO_SPLIT_SEED,
    print_every=PPO_PRINT_EVERY,
    eval_limit=PPO_EVAL_LIMIT,
    device=None,
    save_dir="report_assets/ppo_2x2",
):
    """Train all PPO modes and evaluate on held-out split."""
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    ds_name = resolve_dataset_name(dataset)
    puzzles, solutions = load_sudoku_puzzles_solutions(dataset, nrows=nrows)
    test_puzzles, test_solutions = make_supervised_style_val_lists(
        puzzles,
        solutions,
        train_ratio=train_ratio,
        seed=split_seed,
    )
    if eval_limit is not None:
        print(f"[debug] evaluation is limited to first {eval_limit} test puzzles")

    print(
        f"PPO 2x2 | dataset={ds_name} | train pool n={len(puzzles)} | "
        f"test n={len(test_puzzles)} | episodes/mode={num_episodes} | device={device}"
    )

    results = []
    agents = {}
    rewards_by_mode = {}

    for i, mode in enumerate(modes, start=1):
        cfg = get_ppo_action_config(mode)
        print("\n" + "#" * 72)
        print(f"# [{i}/{len(modes)}] TRAIN {mode}: MRV={cfg['use_mrv_position']}, candidate_mask={cfg['use_valid_candidates']}")
        print("#" * 72)

        agent, rewards = train_ppo_sudoku_progress(
            puzzles,
            solutions,
            num_episodes=num_episodes,
            device=device,
            ppo_mode=mode,
            print_every=print_every,
        )
        agents[mode] = agent
        rewards_by_mode[mode] = rewards

        ckpt_path = save_dir / f"ppo_{mode}_{ds_name}.pth"
        torch.save(agent.policy.state_dict(), ckpt_path)
        print(f"Saved checkpoint: {ckpt_path}")

        print(f"\n--- TEST {mode} ---")
        cell_acc, puzzle_acc = evaluate_ppo_cell_and_puzzle_accuracy(
            agent,
            test_puzzles,
            test_solutions,
            device=device,
            ppo_mode=mode,
            eval_limit=eval_limit,
            show_progress=True,
        )
        print(f"{mode}: cell_accuracy={cell_acc:.4f} | puzzle_accuracy={puzzle_acc:.4f}")

        results.append(
            {
                "mode": mode,
                "dataset": ds_name,
                "nrows": nrows,
                "num_episodes": num_episodes,
                "use_mrv_position": cfg["use_mrv_position"],
                "use_candidate_mask": cfg["use_valid_candidates"],
                "cell_accuracy": cell_acc,
                "puzzle_accuracy": puzzle_acc,
            }
        )

    results_df = pd.DataFrame(results)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = save_dir / f"ppo_2x2_results_{ds_name}_{stamp}.csv"
    json_path = save_dir / f"ppo_2x2_results_{ds_name}_{stamp}.json"
    results_df.to_csv(csv_path, index=False)
    json_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")

    print("\n" + "=" * 72)
    print("PPO 2x2 SUMMARY")
    print("=" * 72)
    print(results_df[["mode", "use_mrv_position", "use_candidate_mask", "cell_accuracy", "puzzle_accuracy"]])
    print(f"\nSaved results:\n  {csv_path}\n  {json_path}")

    plt.figure(figsize=(10, 4))
    for mode, rewards in rewards_by_mode.items():
        if len(rewards) >= 10:
            window = min(50, len(rewards))
            ma = np.convolve(rewards, np.ones(window) / window, mode="valid")
            plt.plot(range(window - 1, window - 1 + len(ma)), ma, label=mode, linewidth=1.5)
        else:
            plt.plot(rewards, label=mode, linewidth=1.5)
    plt.xlabel("Episode")
    plt.ylabel("Reward moving average")
    plt.title("PPO 2x2 training curves")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return results_df, agents, rewards_by_mode


# Run PPO 2x2 batch
ppo_2x2_results_df, ppo_2x2_agents, ppo_2x2_rewards = run_ppo_2x2_train_and_test()
display(ppo_2x2_results_df)


In [ ]:
# compare_ppo_2x2_table(...)
# compare_ppo_2x2_table(ppo_agents_2x2, puzzles, solutions, device, n_eval=500)

# ==========================================
# Supervised loss ablation
# ==========================================
# compare_supervised_loss_ablation(
#     SudokuTransformerTrain,
#     dataset="sudoku-1m",
#     nrows=20_000,
#     num_epoch=50,
#     batch_size=64,
# )

# Or single run:
# SudokuTransformerTrain("sudoku-1m", nrows=20_000, num_epoch=50, loss_on="blanks_only")
# SudokuTransformerTrain("sudoku-1m", nrows=20_000, num_epoch=50, loss_on="all_cells")


### Attention / influence heatmaps

Run `SudokuTransformer` / `SudokuGNNModel` cells first, then train or pass `transformer_model` / `gcn_model`.

- Query cell (5,5) is index `(4, 4)`.
- Transformer: last-layer self-attention over 81 cells.
- GCN: peer graph, gradient attribution, layerwise influence.

Define plotting helpers in the next cell; set `ATTENTION_CFG` and run to save figures under `figs/`.


In [ ]:
# --- Attention / influence visualization (self-contained for Colab) ---
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from matplotlib.patches import Rectangle


def _rc_to_index(row: int, col: int) -> int:
    return row * 9 + col


def puzzle_string_to_batch(puzzle: str, device: torch.device) -> torch.Tensor:
    s = normalize_puzzle_string(puzzle)
    ints = torch.tensor([int(c) for c in s], dtype=torch.long).view(9, 9)
    one_hot = torch.nn.functional.one_hot(ints, num_classes=10).permute(2, 0, 1).float()
    return one_hot.unsqueeze(0).to(device)


def _parse_grid_81(s: str) -> list[list[int]]:
    s = normalize_puzzle_string(s)
    return [[int(s[r * 9 + c]) if s[r * 9 + c] != "0" else 0 for c in range(9)] for r in range(9)]


def _draw_sudoku_board(ax, grid, *, givens=None, title: str = ""):
    ax.set_xlim(0, 9)
    ax.set_ylim(0, 9)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.axis("off")
    for br in range(3):
        for bc in range(3):
            if (br + bc) % 2 == 0:
                ax.add_patch(Rectangle((bc * 3, br * 3), 3, 3, facecolor="#f4f4f4", edgecolor="none", zorder=0))
    for i in range(10):
        lw = 1.8 if i % 3 == 0 else 0.6
        ax.plot([0, 9], [i, i], color="black", linewidth=lw, zorder=2)
        ax.plot([i, i], [0, 9], color="black", linewidth=lw, zorder=2)
    for r in range(9):
        for c in range(9):
            val = grid[r][c]
            if val == 0:
                continue
            bold = givens is not None and givens[r][c] != 0
            ax.text(
                c + 0.5, r + 0.5, str(val), ha="center", va="center",
                fontsize=11, fontweight="bold" if bold else "normal",
                color="#1a1a1a" if bold else "#333333", zorder=3,
            )
    ax.set_title(title, fontsize=10, pad=6)


def _encoder_layer_with_attention(layer: nn.TransformerEncoderLayer, src: torch.Tensor):
    x = src
    x2 = layer.norm1(x)
    attn_out, attn_weights = layer.self_attn(
        x2, x2, x2, need_weights=True, average_attn_weights=False,
    )
    x = x + layer.dropout1(attn_out)
    x2 = layer.norm2(x)
    x = x + layer.dropout2(
        layer.linear2(layer.dropout(layer.activation(layer.linear1(x2))))
    )
    return x, attn_weights


def transformer_self_attention_maps(model: SudokuTransformer, x: torch.Tensor) -> list[torch.Tensor]:
    h = model.embed_tokens(x)
    maps = []
    for layer in model.transformer_encoder.layers:
        h, w = _encoder_layer_with_attention(layer, h)
        maps.append(w.detach().cpu())
    return maps


def query_attention_heatmap(attn_weights, query_index: int, *, combine_heads: str = "max") -> np.ndarray:
    w = attn_weights[0]
    row = w.max(dim=0).values[query_index] if combine_heads == "max" else w.mean(dim=0)[query_index]
    hm = row.numpy().reshape(9, 9)
    lo, hi = float(np.percentile(hm, 5)), float(np.percentile(hm, 95))
    if hi <= lo + 1e-12:
        hi, lo = float(hm.max()), float(hm.min())
    return np.clip((hm - lo) / (hi - lo + 1e-8), 0.0, 1.0)


def gcn_peer_structural_map(model: SudokuGNNModel, query_index: int) -> np.ndarray:
    row = model.adj[query_index].detach().cpu().numpy().reshape(9, 9)
    m = row.max()
    return row / m if m > 0 else row


def gcn_gradient_attribution(model: SudokuGNNModel, x: torch.Tensor, query_row: int, query_col: int) -> np.ndarray:
    model.eval()
    x = x.clone().detach().requires_grad_(True)
    logits = model(x)
    pred = logits[0, :, query_row, query_col].argmax().item()
    score = logits[0, pred, query_row, query_col]
    model.zero_grad(set_to_none=True)
    score.backward()
    grad = x.grad[0].abs().sum(dim=0).detach().cpu().numpy()
    m = grad.max()
    return grad / m if m > 0 else grad


def gcn_layerwise_influence(model: SudokuGNNModel, x: torch.Tensor, query_index: int) -> np.ndarray:
    model.eval()
    b = x.size(0)
    h = x.permute(0, 2, 3, 1).reshape(b, 81, 10)
    h = model.input_lin(h)
    adj = model.adj.to(dtype=h.dtype, device=h.device)
    influence = torch.zeros(81, device=h.device, dtype=h.dtype)
    with torch.no_grad():
        for lin, ln in zip(model.layers, model.norms):
            msg = torch.einsum("ij,bjf->bif", adj, h)
            msg = ln(msg)
            influence = influence + (adj[query_index].unsqueeze(0).unsqueeze(-1) * h).abs().sum(dim=-1)[0]
            h = h + torch.relu(lin(msg))
    arr = influence.detach().cpu().numpy()
    arr = arr / (arr.max() + 1e-8)
    return arr.reshape(9, 9)


def _plot_heatmap_ax(ax, hm, query_row, query_col, title: str):
    ax.imshow(hm, cmap="YlOrRd", vmin=0.0, vmax=1.0, aspect="equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for i in range(10):
        lw = 1.4 if i % 3 == 0 else 0.35
        ax.axhline(i - 0.5, color="k", linewidth=lw)
        ax.axvline(i - 0.5, color="k", linewidth=lw)
    ax.scatter([query_col], [query_row], s=110, facecolors="none", edgecolors="#1a5276", linewidths=2)
    ax.set_title(title, fontsize=9)
    ax.invert_yaxis()


def plot_attention_figures(
    puzzle: str,
    transformer_model: SudokuTransformer,
    gcn_model: SudokuGNNModel,
    *,
    query_row: int = 4,
    query_col: int = 4,
    out_dir: str | Path = "figs",
    show_inline: bool = True,
) -> dict[str, Path]:
    """Generate report figures; returns paths of saved files."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    device = next(transformer_model.parameters()).device
    transformer_model.eval()
    gcn_model.eval()

    x = puzzle_string_to_batch(puzzle, device)
    q_idx = _rc_to_index(query_row, query_col)
    qr, qc = query_row, query_col
    grid = _parse_grid_81(puzzle)

    with torch.no_grad():
        attn_layers = transformer_self_attention_maps(transformer_model, x)
    tfm_last = query_attention_heatmap(attn_layers[-1], q_idx)

    struct = gcn_peer_structural_map(gcn_model, q_idx)
    with torch.enable_grad():
        grad_map = gcn_gradient_attribution(gcn_model, x, qr, qc)
    layer_inf = gcn_layerwise_influence(gcn_model, x, q_idx)

    paths = {}

    # Combined 2x2 (recommended for report)
    fig, axes = plt.subplots(2, 2, figsize=(7.2, 7.0))
    _draw_sudoku_board(axes[0, 0], grid, givens=grid, title="Input")
    for ax, hm, title in [
        (axes[0, 1], tfm_last, "Transformer (last layer)"),
        (axes[1, 0], grad_map, "GCN (gradient attr.)"),
        (axes[1, 1], layer_inf, "GCN (layerwise infl.)"),
    ]:
        _plot_heatmap_ax(ax, hm, qr, qc, title)
    fig.suptitle(f"Query cell ({qr+1},{qc+1}) influence on the board", fontsize=11)
    fig.subplots_adjust(hspace=0.22, wspace=0.12, right=0.88)
    cbar = fig.colorbar(axes[1, 1].images[0], ax=axes.ravel().tolist(), fraction=0.03, pad=0.02)
    cbar.set_label("normalized weight")
    for ext in ("png", "pdf"):
        p = out_dir / f"attention_combined.{ext}"
        fig.savefig(p, dpi=220, bbox_inches="tight")
        paths[f"combined_{ext}"] = p
    if show_inline:
        plt.show()
    else:
        plt.close(fig)

    # Transformer-only panel
    fig2, axes2 = plt.subplots(1, 3, figsize=(9.5, 3.4))
    _draw_sudoku_board(axes2[0], grid, givens=grid, title="Input")
    _plot_heatmap_ax(axes2[1], tfm_last, qr, qc, f"Transformer L{len(attn_layers)}")
    mean_map = np.mean([query_attention_heatmap(w, q_idx) for w in attn_layers], axis=0)
    _plot_heatmap_ax(axes2[2], mean_map, qr, qc, f"Transformer mean L1-L{len(attn_layers)}")
    fig2.suptitle("Transformer self-attention", fontsize=10)
    fig2.tight_layout()
    for ext in ("png", "pdf"):
        p = out_dir / f"attention_transformer.{ext}"
        fig2.savefig(p, dpi=220, bbox_inches="tight")
        paths[f"transformer_{ext}"] = p
    if show_inline:
        plt.show()
    else:
        plt.close(fig2)

    # GCN panel
    fig3, axes3 = plt.subplots(1, 4, figsize=(12.5, 3.4))
    _draw_sudoku_board(axes3[0], grid, givens=grid, title="Input")
    for ax, hm, title in zip(
        axes3[1:],
        [struct, grad_map, layer_inf],
        ["GCN peers", "GCN grad attr.", "GCN layerwise"],
    ):
        _plot_heatmap_ax(ax, hm, qr, qc, title)
    fig3.suptitle("GCN influence maps", fontsize=10)
    fig3.tight_layout()
    for ext in ("png", "pdf"):
        p = out_dir / f"attention_gcn.{ext}"
        fig3.savefig(p, dpi=220, bbox_inches="tight")
        paths[f"gcn_{ext}"] = p
    if show_inline:
        plt.show()
    else:
        plt.close(fig3)

    print("Saved:")
    for k, p in paths.items():
        print(f"  {k}: {p}")
    return paths


In [ ]:
# ATTENTION_CFG
ATTENTION_CFG = {
  # example puzzle
  "puzzle": "006054102020000048008700030703410600054000000201930000100208000007006300000170405",
  "query_row": 4,
  "query_col": 4,
  # pass trained models from *Train
  "transformer_model": None,
  "gcn_model": None,
  # mini_train_if_missing: quick demo only
  "mini_train_if_missing": False,
  "mini_nrows": 3000,
  "mini_epochs": 8,
  # out_dir for saved figures
  "out_dir": "figs",
}

# resolve models
def _resolve_trained_model(maybe_model, label: str, build_model):
    if maybe_model is not None:
        return maybe_model
    if not ATTENTION_CFG["mini_train_if_missing"]:
        raise ValueError(
            f"Set ATTENTION_CFG['{label}'] to a trained model, or mini_train_if_missing=True for demo"
        )
    print(f"Mini-training {label} ({ATTENTION_CFG['mini_nrows']} rows, {ATTENTION_CFG['mini_epochs']} epochs)…")
    train_ds, val_ds, name = prepare_supervised_splits(
        "sudoku-1m", nrows=ATTENTION_CFG["mini_nrows"], split_seed=42,
    )
    m = build_model()
    tr = SudokuTrainer(
        m, train_ds, val_ds,
        batch_size=64,
        num_epoch=ATTENTION_CFG["mini_epochs"],
        save_dir=f"./checkpoint/viz_{label}_{name}_allcells",
        loss_on="all_cells",
    )
    tr.train()
    return tr.model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tfm_viz = _resolve_trained_model(
    ATTENTION_CFG["transformer_model"],
    "transformer_model",
    lambda: SudokuTransformer(128, 4, 4),
).to(device)
gcn_viz = _resolve_trained_model(
    ATTENTION_CFG["gcn_model"],
    "gcn_model",
    lambda: SudokuGNNModel(hidden_dim=128, num_layers=6),
).to(device)

saved = plot_attention_figures(
    ATTENTION_CFG["puzzle"],
    tfm_viz,
    gcn_viz,
    query_row=ATTENTION_CFG["query_row"],
    query_col=ATTENTION_CFG["query_col"],
    out_dir=ATTENTION_CFG["out_dir"],
    show_inline=True,
)

# Optional Colab download
try:
    from google.colab import files
    files.download(str(saved["combined_png"]))
except ImportError:
    pass
